<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/cosineSimilarityOfEmbeddings/Phi_3_mini_4k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q bitsandbytes>=0.46.1
!pip install -U torchao
!pip install -U sentence-transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
import pandas as pd
import torch
import re
from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, TaskType

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
sentenceTransformers = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
device = torch.device("cuda")
device

device(type='cuda')

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

fineTunedModel = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/mental-health-phi3mini4k"
)

ogModel = AutoModelForCausalLM.from_pretrained(
    "unsloth/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

ogModel.to(device)
fineTunedModel.to(device)

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_feature

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

TokenizersBackend(name_or_path='unsloth/Phi-3-mini-4k-instruct', vocab_size=32000, model_max_length=4096, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<|placeholder2|>", rstrip=Tr

In [ ]:
messages = [
    # {"role": "assistant", "content": "I feel completely lost after my dog died. What should I do to cope day to day?"},
    {"role": "user", "content": input("")}
]

I feel completely lost after my dog died. What should I do to cope day to day?


In [ ]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
	return_dict=True,
	return_tensors="pt").to(device)
inputs

{'input_ids': tensor([[32010,   306,  4459,  6446,  5714,  1156,   590, 11203,  6423, 29889,
          1724,   881,   306,   437,   304,  1302,   412,  2462,   304,  2462,
         29973, 32007, 32001]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [ ]:
# model_inputs = encoded_message.to(device)
# model_inputs

In [ ]:
fineTunedOutputs = fineTunedModel.generate(**inputs, max_new_tokens=150, max_length=150, num_return_sequences=3, do_sample=True)
fineTunedOutputs

Both `max_new_tokens` (=150) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[32010,   306,  4459,  6446,  5714,  1156,   590, 11203,  6423, 29889,
          1724,   881,   306,   437,   304,  1302,   412,  2462,   304,  2462,
         29973, 32007, 32001,   306,   626, 19781,  7423,   363,   596,  6410,
         29889, 10061,   292,   411,   278,  4892,   310,   263,  5697,   338,
          1422,   515, 19035,   263,  5199, 18708, 29892,   541,   596, 21737,
           526, 18018,  2854, 29889,  2266,   526,   777, 16650,   583,   304,
          1371,   366,  1302,   412,   411,   596,  6410,  2462,   491,  2462,
         29901,    13,    13,    13, 29896, 29889,  3579, 15930,  3575,  1311,
           304,  1632,  2418, 29901,  1068,  1670,   338,   694,   731,  5335,
          5570,   363,   867,  2575, 29892,   322, 14332, 17766,   372, 17587,
         29889, 25538,  7535, 10751,   304,  4459, 14610,  2264, 29892, 27343,
         29892, 14679, 29892,   470,  1584, 18892,   363, 23023,  1080,   366,
          1795,   316, 22580,   443, 16044,   519, 2

In [ ]:
ogModelOutputs = ogModel.generate(**inputs, max_new_tokens=150, max_length=150, num_return_sequences=3, do_sample=True)
ogModelOutputs

Both `max_new_tokens` (=150) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[32010,   306,  4459,  6446,  5714,  1156,   590, 11203,  6423, 29889,
          1724,   881,   306,   437,   304,  1302,   412,  2462,   304,  2462,
         29973, 32007, 32001,   306, 30010, 29885,  7423,   304,  8293,  1048,
           596,  6410, 29889, 10061,   292,   411,   278,  4892,   310,   263,
          5697,   338,  2600,   618,   368,  5189, 29889,  2266,   526,   777,
          2462, 29899,   517,  2462,  5614,   292, 16650,   583,   366,  1795,
          2050,   304,  1371,   366,  1549,   445,   931, 29901,    13,    13,
            13, 29896, 29889, 29408,  7535,   304,   867,  2418, 29901, 25538,
          7535, 10751,   304,  4459, 14610,   322, 18145,  5485,   393,   372,
         30010, 29879, 20759,   304,   367, 24081,   300,  1048,   596, 11203,
         30010, 29879,  4892, 29889,    13,    13, 29906, 29889,  2661,   370,
          1674,  6745,  1475, 29901,  2973,   263, 11203,  7695, 29892,  1432,
          3250,  6745,  1475,   674,   367,   766, 1

In [ ]:
print(tokenizer.decode(fineTunedOutputs))
fineTunedResponse = tokenizer.decode(fineTunedOutputs[0])
fineTunedResponse

['<|user|> I feel completely lost after my dog died. What should I do to cope day to day?<|end|><|assistant|> I am truly sorry for your loss. Coping with the death of a pet is different from losing a human companion, but your feelings are equally valid. Here are some strategies to help you cope with your loss day by day:\n\n\n1. **Allow Yourself to Grieve:** There is no set timeline for grief, and everyone handles it differently. Give yourself permission to feel sadness, anger, confusion, or even relief for emotions you might deemed unacceptable.\n\n\n2. **Create a Memory Box:** A memory box can be a tangible representation of your bond with your dog. Collect photos, favorite toys, a collar, or anything that brings you joy or', "<|user|> I feel completely lost after my dog died. What should I do to cope day to day?<|end|><|assistant|> I am deeply sorry to hear about your loss. It's completely natural to feel lost and grief-stricken when a beloved pet passes away. Here are some ways you

'<|user|> I feel completely lost after my dog died. What should I do to cope day to day?<|end|><|assistant|> I am truly sorry for your loss. Coping with the death of a pet is different from losing a human companion, but your feelings are equally valid. Here are some strategies to help you cope with your loss day by day:\n\n\n1. **Allow Yourself to Grieve:** There is no set timeline for grief, and everyone handles it differently. Give yourself permission to feel sadness, anger, confusion, or even relief for emotions you might deemed unacceptable.\n\n\n2. **Create a Memory Box:** A memory box can be a tangible representation of your bond with your dog. Collect photos, favorite toys, a collar, or anything that brings you joy or'

In [ ]:
print(tokenizer.decode(ogModelOutputs))
ogResponse = tokenizer.decode(ogModelOutputs[0])
ogResponse

["<|user|> I feel completely lost after my dog died. What should I do to cope day to day?<|end|><|assistant|> I’m sorry to hear about your loss. Coping with the death of a pet is profoundly difficult. Here are some day-to day coping strategies you might consider to help you through this time:\n\n\n1. Allow yourself to grieve: Give yourself permission to feel sad and acknowledge that it’s okay to be upset about your dog’s death.\n\n2. Establish routines: With a dog gone, everyday routines will be disrupted. Creating a new structure can give you a sense of normalcy.\n\n3. Find comfort in rituals: Consider establishing a new ritual to honor your dog's memory, such as looking at pictures, planning", "<|user|> I feel completely lost after my dog died. What should I do to cope day to day?<|end|><|assistant|> Coping with the loss of a dog can be incredibly difficult, as pets are often considered part of the family. It's important to remember that it'ieves no one to grieve. Here are some daily

"<|user|> I feel completely lost after my dog died. What should I do to cope day to day?<|end|><|assistant|> I’m sorry to hear about your loss. Coping with the death of a pet is profoundly difficult. Here are some day-to day coping strategies you might consider to help you through this time:\n\n\n1. Allow yourself to grieve: Give yourself permission to feel sad and acknowledge that it’s okay to be upset about your dog’s death.\n\n2. Establish routines: With a dog gone, everyday routines will be disrupted. Creating a new structure can give you a sense of normalcy.\n\n3. Find comfort in rituals: Consider establishing a new ritual to honor your dog's memory, such as looking at pictures, planning"

In [ ]:
encodedFineTuned = sentenceTransformers.encode(fineTunedResponse)
encodedFineTuned.shape

(384,)

In [ ]:
encodedOg = sentenceTransformers.encode(ogResponse)
encodedOg.shape

(384,)

In [ ]:
cosine_similarity([encodedFineTuned], [encodedOg])

array([[0.92300093]], dtype=float32)

In [ ]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/normalized_context_and_response.csv")

In [ ]:
def remove_special_tokens(text):
    text = re.sub(r'<\|.*?\|>', '', text)
    return text.strip()

In [ ]:
contexts = dataset['context'].apply(remove_special_tokens).astype("str").values
contexts[:1]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone"],
      dtype=object)

In [ ]:
responses = dataset['response'].astype("str").apply(remove_special_tokens).values
responses[:1]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today"],
      dtype=object)

In [ ]:
def combineText(example):
  messages = [
      {"role": "user", "content": example['contexts']},
      {"role": "assistant", "content": example['responses']}
  ]
  formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return {"text": formatted_text}

In [ ]:
def encode(example):
  return tokenizer(example['text'], truncation=True, padding=True, max_length=128)

In [ ]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [ ]:
datasets = Dataset.from_dict({
    "contexts": contexts,
    "responses": responses
})
datasets

Dataset({
    features: ['contexts', 'responses'],
    num_rows: 3512
})

In [ ]:
datasets_split = datasets.train_test_split( test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['contexts', 'responses'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['contexts', 'responses'],
        num_rows: 703
    })
})

In [ ]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['contexts', 'responses'],
    num_rows: 2809
})

In [ ]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['contexts', 'responses'],
    num_rows: 703
})

In [ ]:
trainSet = trainSet.map(combineText)

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [ ]:
testSet = testSet.map(combineText)

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

In [ ]:
trainSet = trainSet.map(encode, batched=True)

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [ ]:
testSet = testSet.map(encode, batched=True)

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

In [ ]:
trainSet = trainSet.map(add_labels)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['contexts', 'responses', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [ ]:
testSet = testSet.map(add_labels)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['contexts', 'responses', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [ ]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=1, # Further reduced batch size to prevent OOM
    per_device_eval_batch_size=1, # Reduced eval batch size for consistency
    gradient_accumulation_steps=8, # Use gradient accumulation to achieve an effective batch size of 1 * 8 = 8
    eval_strategy='steps',
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    bf16=True, # Use bfloat16 for better memory stability and efficiency on T4
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    max_grad_norm = 1.0
)

In [ ]:
model.add_adapter(lora_config, adapter_name="my_adapter")

NameError: name 'model' is not defined

In [ ]:
trainer = Trainer(
    model=model,
    args=trainingArgs,
    train_dataset=trainSet.select(range(100)),  # Using a small subset for faster training
    eval_dataset=testSet.select(range(80))    # Using a small subset for faster evaluation
)

In [ ]:
trainer.evaluate(testSet.select(range(80)))

In [ ]:
trainer.predict(testSet.select(range(80)))

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/mental-health-phi3mini4k")

In [ ]:
trainer.evaluate(testSet.select(range(80)))

In [ ]:
trainer.predict(testSet.select(range(80)))